# Trabajo Práctico ML 1 — Preprocesamiento y EDA
## Clínica VitalPlus

**Asignatura:** Machine Learning  
**Herramienta:** R en Jupyter (Google Colab, kernel R / IRkernel)

### Índice
1. [3.1 Comprensión del negocio](#31)
2. [Setup — paquetes, repositorio y datos](#setup)
3. [3.2 Comprensión de los datos](#32)
4. [3.3 Preparación de los datos](#33)
5. [3.4 Análisis Exploratorio de Datos (EDA)](#34)
6. [3.5 Cierre](#35)

> **Colab:** Runtime → Change runtime type → **R**. Reemplazar `REPO_URL` en el setup con la URL de tu repositorio GitHub.


## 3.1 — Comprensión del negocio

Clínica VitalPlus es un centro de diagnóstico médico con sedes en varias ciudades de Colombia. La dirección quiere entender qué factores se asocian con la **duración**, el **costo** y la **satisfacción** de los exámenes, como paso previo a construir un modelo de regresión que estime el costo de un examen a partir de su duración.

**Objetivo de negocio:** Identificar patrones en los datos operativos que permitan anticipar el costo de los exámenes según su duración, mejorar la planificación de recursos y la experiencia del paciente.

**Objetivo de este análisis exploratorio:** Evaluar la calidad del dataset, limpiarlo de forma reproducible y explorar la relación duración–costo (y otras variables relevantes) para preparar el terreno del modelo de regresión.

**Criterios de éxito medibles:**
1. Dataset procesado sin valores nulos en `duracion_minutos` y `costo_examen` (variables del futuro modelo).
2. Relación duración–costo cuantificada (correlación + gráfico de dispersión) y documentada con interpretación de negocio.


## Setup — paquetes, repositorio y datos

Antes del análisis instalamos las librerías necesarias, clonamos el repositorio con el dataset y definimos funciones auxiliares que reutilizaremos en las secciones 3.2 y 3.3.


In [ ]:
paquetes <- c("tidyverse", "lubridate", "skimr", "corrplot", "stringi")
for (pkg in paquetes) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lubridate)
library(skimr)
library(corrplot)
library(stringi)


In [ ]:
# Reemplazar con la URL real del repositorio antes de ejecutar en Colab
REPO_URL <- "https://github.com/<tu-usuario>/ml-p1-eda-preprocessing.git"
REPO_DIR <- "ml-p1-eda-preprocessing"

if (!dir.exists(REPO_DIR)) {
  system(paste("git clone", REPO_URL))
}
setwd(REPO_DIR)

datos_crudos <- readr::read_csv(
  "data/vitalplus_pacientes.csv",
  show_col_types = FALSE
)


In [ ]:
reporte_nulos <- function(df) {
  df %>%
    summarise(across(everything(), ~ sum(is.na(.x)))) %>%
    pivot_longer(everything(), names_to = "variable", values_to = "nulos") %>%
    mutate(porcentaje = round(100 * nulos / nrow(df), 2)) %>%
    arrange(desc(nulos))
}

limpiar_costo <- function(x) {
  x <- as.character(x)
  x <- str_replace_all(x, "[$\\s]", "")
  x <- str_replace_all(x, "\\.", "")
  as.numeric(x)
}

winsorizar_iqr <- function(x, mult = 1.5) {
  q <- quantile(x, probs = c(0.25, 0.75), na.rm = TRUE)
  iqr_val <- q[2] - q[1]
  lim_inf <- q[1] - mult * iqr_val
  lim_sup <- q[2] + mult * iqr_val
  recortados <- sum(x < lim_inf | x > lim_sup, na.rm = TRUE)
  list(
    valor = pmin(pmax(x, lim_inf), lim_sup),
    lim_inf = lim_inf,
    lim_sup = lim_sup,
    recortados = recortados
  )
}

normalizar_texto <- function(x) {
  x <- str_trim(as.character(x))
  x <- stri_trans_general(x, "Latin-ASCII")
  str_to_lower(x)
}


## 3.2 — Comprensión de los datos

En esta sección describimos la estructura del dataset y elaboramos un reporte de calidad por columna: valores nulos, filas duplicadas, valores fuera de rango u outliers, e inconsistencias en categorías y formatos. Todo el análisis se realiza sobre `datos_crudos` sin modificar los datos.


In [ ]:
cat("Registros:", nrow(datos_crudos), "\n")
cat("Variables:", ncol(datos_crudos), "\n")
cat("Nombres:", paste(names(datos_crudos), collapse = ", "), "\n\n")
glimpse(datos_crudos)
skim(datos_crudos)


El dataset contiene **576 registros** y **11 variables**. Al leer el CSV, `costo_examen` aparece como texto (character) porque algunos valores incluyen símbolos de moneda y separadores de miles. `fecha_cita` también es texto con formatos de fecha mezclados. Las variables numéricas (`edad`, `duracion_minutos`, `calificacion_satisfaccion`) tienen valores nulos parciales que debemos cuantificar.


In [ ]:
tabla_nulos <- reporte_nulos(datos_crudos)
print(tabla_nulos)


Las columnas con más valores faltantes son `costo_examen` (~6%) y `duracion_minutos` (~4%). También hay nulos en `edad`, `canal_remision` y `calificacion_satisfaccion` (~3% cada una). Las variables `id_paciente`, `fecha_cita`, `dias_espera_cita` y `reingreso_30dias` no tienen nulos. Estos vacíos deben tratarse en la fase 3.3, con especial atención a duración y costo por ser las variables del futuro modelo.


In [ ]:
duplicados_exactos <- sum(duplicated(datos_crudos))
duplicados_id <- sum(duplicated(datos_crudos$id_paciente))

cat("Filas duplicadas (exactas):", duplicados_exactos, "\n")
cat("id_paciente repetidos:", duplicados_id, "\n")

datos_crudos %>%
  count(id_paciente) %>%
  filter(n > 1) %>%
  head(10)


Se detectan **16 filas duplicadas** y **16 identificadores de paciente repetidos**. Esto sugiere registros duplicados por error de captura, no pacientes distintos con el mismo ID. En el preprocesamiento eliminaremos duplicados conservando la primera aparición de cada `id_paciente`.


In [ ]:
costo_num_crudo <- limpiar_costo(datos_crudos$costo_examen)

boxplot(
  datos_crudos$duracion_minutos,
  main = "duracion_minutos (crudo)",
  ylab = "minutos"
)
boxplot(
  costo_num_crudo,
  main = "costo_examen (crudo, limpiado)",
  ylab = "COP"
)

fuera_rango_cal <- sum(
  datos_crudos$calificacion_satisfaccion < 1 |
    datos_crudos$calificacion_satisfaccion > 5,
  na.rm = TRUE
)
fuera_rango_edad <- sum(
  datos_crudos$edad < 0 | datos_crudos$edad > 120,
  na.rm = TRUE
)

cat("Calificación fuera de 1-5:", fuera_rango_cal, "\n")
cat("Edad fuera de 0-120:", fuera_rango_edad, "\n")
cat("Max duracion_minutos:", max(datos_crudos$duracion_minutos, na.rm = TRUE), "\n")
cat("Max costo_examen:", max(costo_num_crudo, na.rm = TRUE), "\n")


Los boxplots revelan valores extremos en **duración** (hasta 400 minutos) y **costo** (valores muy superiores al rango típico). Además hay **5 calificaciones fuera del rango 1–5** y **4 edades fuera de 0–120 años**, que son errores de captura. En 3.3 corregiremos rangos inválidos y aplicaremos winsorización IQR a duración y costo para reducir el efecto de outliers en la futura regresión, sin eliminar filas.


In [ ]:
cat("--- sede (valores únicos) ---\n")
datos_crudos %>% count(sede, sort = TRUE) %>% print(n = 20)

cat("\n--- tipo_examen (valores únicos) ---\n")
datos_crudos %>% count(tipo_examen, sort = TRUE) %>% print(n = 20)

cat("\n--- canal_remision (valores únicos) ---\n")
datos_crudos %>% count(canal_remision, sort = TRUE) %>% print(n = 20)


Las tres variables categóricas muestran **inconsistencias de formato**: mayúsculas/minúsculas (`BOGOTA` vs `Bogotá`), acentos (`Bogota` vs `Bogotá`), abreviaturas (`Lab` vs `Laboratorio`, `Eco` vs `Ecografía`, `Rx` vs `Radiografía`) y variantes de canal (`EPS` vs `E.P.S.`, `Prepagada` vs `Medicina prepagada`). En 3.3 aplicaremos un diccionario de mapeo explícito hacia categorías canónicas.


In [ ]:
cat("Muestra de formatos en fecha_cita:\n")
datos_crudos %>%
  distinct(fecha_cita) %>%
  slice_head(n = 15)

costo_texto <- as.character(datos_crudos$costo_examen)
formato_moneda <- sum(grepl("[$]", costo_texto), na.rm = TRUE)
cat("\nRegistros de costo con símbolo $:", formato_moneda, "\n")

cat("\nMuestra de costos con formato no estándar:\n")
datos_crudos %>%
  filter(grepl("[$]", as.character(costo_examen))) %>%
  select(id_paciente, costo_examen) %>%
  head(10)


### Resumen del reporte de calidad

| Problema | Columnas afectadas | Magnitud |
|----------|---------------------|----------|
| Valores nulos | costo, duración, edad, sede, tipo_examen, canal, calificación | 1–6% por columna |
| Duplicados | id_paciente | 16 registros |
| Outliers | duracion_minutos, costo_examen | Valores extremos visibles en boxplots |
| Fuera de rango | calificacion_satisfaccion, edad | 5 y 4 registros respectivamente |
| Categorías inconsistentes | sede, tipo_examen, canal_remision | Múltiples variantes por categoría |
| Formatos mixtos | fecha_cita, costo_examen | Varios formatos de fecha; costos con `$` y puntos de miles |

Este diagnóstico guía el pipeline de limpieza en la sección 3.3.


## 3.3 — Preparación de los datos

A partir de los problemas detectados en 3.2, aplicamos un pipeline de limpieza secuencial. Cada decisión incluye la técnica elegida y la alternativa descartada. El resultado final se guarda en `datos_limpios` y se exporta como CSV.


### 3.3.1 — Duplicados

**Problema:** 16 registros con `id_paciente` repetido.  
**Técnica:** Eliminar duplicados por `id_paciente`, conservando la primera aparición (`distinct(.keep_all = TRUE)`).  
**Alternativa descartada:** Mantener duplicados o promediar valores — distorsionaría conteos y relaciones en el EDA.


In [ ]:
datos_limpios <- datos_crudos %>%
  distinct(id_paciente, .keep_all = TRUE)

cat("Filas antes:", nrow(datos_crudos), "→ después:", nrow(datos_limpios), "\n")


### 3.3.2 — Categorías inconsistentes

**Problema:** Variantes de sede, tipo de examen y canal de remisión (mayúsculas, acentos, abreviaturas).  
**Técnica:** Normalizar texto (trim, minúsculas, sin acentos) y mapear con diccionario explícito hacia categorías canónicas. Valores nulos o no reconocidos → `"Desconocido"`.  
**Alternativa descartada:** Imputar con la moda — sesgaría hacia Bogotá/EPS que ya son las categorías más frecuentes.


In [ ]:
map_sede <- c(
  "bogota" = "Bogotá", "cali" = "Cali",
  "medellin" = "Medellín", "bucaramanga" = "Bucaramanga"
)
map_tipo <- c(
  "laboratorio" = "Laboratorio", "lab" = "Laboratorio",
  "ecografia" = "Ecografía", "eco" = "Ecografía",
  "radiografia" = "Radiografía", "rx" = "Radiografía",
  "resonancia" = "Resonancia", "rmn" = "Resonancia",
  "tomografia" = "Tomografía", "tac" = "Tomografía"
)
map_canal <- c(
  "eps" = "EPS", "e.p.s." = "EPS",
  "particular" = "Particular",
  "remitido" = "Remitido",
  "prepagada" = "Medicina prepagada",
  "medicina prepagada" = "Medicina prepagada"
)

datos_limpios <- datos_limpios %>%
  mutate(
    sede_norm = normalizar_texto(sede),
    tipo_norm = normalizar_texto(tipo_examen),
    canal_norm = normalizar_texto(canal_remision),
    sede = map_sede[sede_norm],
    tipo_examen = map_tipo[tipo_norm],
    canal_remision = map_canal[canal_norm]
  ) %>%
  mutate(
    sede = if_else(is.na(sede), "Desconocido", sede),
    tipo_examen = if_else(is.na(tipo_examen), "Desconocido", tipo_examen),
    canal_remision = if_else(is.na(canal_remision), "Desconocido", canal_remision)
  ) %>%
  select(-sede_norm, -tipo_norm, -canal_norm)

datos_limpios %>% count(sede, sort = TRUE)
datos_limpios %>% count(tipo_examen, sort = TRUE)
datos_limpios %>% count(canal_remision, sort = TRUE)


### 3.3.3 — Formatos de fecha inconsistentes

**Problema:** `fecha_cita` mezcla formatos (`YYYY-MM-DD`, `DD-Mon-YYYY`, `DD/MM/YYYY`, etc.).  
**Técnica:** `parse_date_time()` con múltiples órdenes (`ymd`, `dmy`, `mdy`, `bmdy`, `bdmy`).  
**Alternativa descartada:** `as.Date()` con un solo formato — falla con formatos heterogéneos.


In [ ]:
datos_limpios <- datos_limpios %>%
  mutate(
    fecha_cita = parse_date_time(
      fecha_cita,
      orders = c("ymd", "dmy", "mdy", "bmdy", "bdmy"),
      quiet = TRUE
    )
  )

fechas_na <- sum(is.na(datos_limpios$fecha_cita))
cat("Fechas no parseadas:", fechas_na, "\n")


### 3.3.4 — costo_examen almacenado como texto

**Problema:** Valores con `$`, espacios y punto como separador de miles (ej. `$ 85.000`).  
**Técnica:** Función `limpiar_costo()` — elimina `$` y espacios, quita puntos de miles, convierte a numérico.  
**Alternativa descartada:** Eliminar filas no parseables — perderíamos ~54 registros con información útil.


In [ ]:
datos_limpios <- datos_limpios %>%
  mutate(costo_examen = limpiar_costo(costo_examen))

cat("Nulos en costo tras limpieza:", sum(is.na(datos_limpios$costo_examen)), "\n")
summary(datos_limpios$costo_examen)


### 3.3.5 — Valores fuera de rango

**Problema:** Calificaciones fuera de 1–5 y edades fuera de 0–120.  
**Técnica:** Reemplazar valores inválidos por `NA` para imputar después.  
**Alternativa descartada:** Conservar valores erróneos — distorsionarían estadísticas y correlaciones.


In [ ]:
datos_limpios <- datos_limpios %>%
  mutate(
    calificacion_satisfaccion = if_else(
      calificacion_satisfaccion < 1 | calificacion_satisfaccion > 5,
      NA_real_, calificacion_satisfaccion
    ),
    edad = if_else(edad < 0 | edad > 120, NA_real_, edad)
  )


### 3.3.6 — Valores nulos

**Problema:** Nulos restantes en edad, calificación, sede, tipo_examen, canal, duración y costo.  
**Técnicas:**
- `edad` y `calificacion_satisfaccion`: mediana global (robusta ante outliers).
- `sede`, `tipo_examen`, `canal_remision`: categoría `"Desconocido"`.
- `duracion_minutos` y `costo_examen`: mediana **por `tipo_examen`** (un laboratorio no debe imputarse con la duración de una resonancia).

**Alternativa descartada para duración/costo:** mediana global — ignoraría que distintos tipos de examen tienen duraciones y costos muy diferentes.


In [ ]:
mediana_edad <- median(datos_limpios$edad, na.rm = TRUE)
mediana_cal <- median(datos_limpios$calificacion_satisfaccion, na.rm = TRUE)

medias_por_tipo <- datos_limpios %>%
  group_by(tipo_examen) %>%
  summarise(
    med_dur = median(duracion_minutos, na.rm = TRUE),
    med_costo = median(costo_examen, na.rm = TRUE),
    .groups = "drop"
  )

datos_limpios <- datos_limpios %>%
  left_join(medias_por_tipo, by = "tipo_examen") %>%
  mutate(
    edad = if_else(is.na(edad), mediana_edad, edad),
    calificacion_satisfaccion = if_else(
      is.na(calificacion_satisfaccion), mediana_cal, calificacion_satisfaccion
    ),
    sede = if_else(is.na(sede), "Desconocido", sede),
    tipo_examen = if_else(is.na(tipo_examen), "Desconocido", tipo_examen),
    canal_remision = if_else(is.na(canal_remision), "Desconocido", canal_remision),
    duracion_minutos = if_else(is.na(duracion_minutos), med_dur, duracion_minutos),
    costo_examen = if_else(is.na(costo_examen), med_costo, costo_examen)
  ) %>%
  select(-med_dur, -med_costo)

print(reporte_nulos(datos_limpios))


### 3.3.7 — Outliers en duracion_minutos y costo_examen

**Problema:** Valores extremos en duración (hasta 400 min) y costo que pueden distorsionar una regresión lineal.  
**Técnica:** Winsorización IQR (1.5×): recortar valores por debajo de Q1−1.5·IQR y por encima de Q3+1.5·IQR al límite correspondiente.  
**Alternativas descartadas:**
- *Eliminar filas outlier:* reduce el tamaño muestral (~560 filas).
- *Conservar sin tratar:* apalanca coeficientes de la regresión futura.
- *Log-transform aquí:* cambia la interpretabilidad; se evaluará en 3.5 para el modelado.


In [ ]:
res_dur <- winsorizar_iqr(datos_limpios$duracion_minutos)
res_costo <- winsorizar_iqr(datos_limpios$costo_examen)

cat("Duración — recortados:", res_dur$recortados,
    "| límite inf:", res_dur$lim_inf, "| límite sup:", res_dur$lim_sup, "\n")
cat("Costo — recortados:", res_costo$recortados,
    "| límite inf:", res_costo$lim_inf, "| límite sup:", res_costo$lim_sup, "\n")

datos_limpios <- datos_limpios %>%
  mutate(
    duracion_minutos = res_dur$valor,
    costo_examen = res_costo$valor
  )


In [ ]:
readr::write_csv(datos_limpios, "data/vitalplus_pacientes_procesado.csv")
cat("Exportado: data/vitalplus_pacientes_procesado.csv\n")
cat("Filas finales:", nrow(datos_limpios), "\n")


### Resumen del preprocesamiento

| Aspecto | Antes (crudo) | Después (limpio) |
|---------|---------------|------------------|
| Filas | 576 | 560 (16 duplicados eliminados) |
| Nulos en duración/costo | Sí | No |
| Categorías | Inconsistentes | 5 sedes, 6 tipos, 5 canales canónicos |
| Outliers duración/costo | Extremos visibles | Winsorizados (IQR 1.5×) |

El dataset procesado está listo para el EDA en 3.4 y como base para el futuro modelo de regresión.


## 3.4 — Análisis Exploratorio de Datos (EDA)

Exploramos la distribución de variables individuales (univariado), relaciones entre variables (bivariado) y la matriz de correlación. El foco principal es la relación **duración–costo**, anticipo del futuro modelo de regresión.


### 3.4.1 — Distribución de duracion_minutos

La duración del examen es la variable independiente del futuro modelo. Analizamos su distribución para entender si los exámenes son mayormente cortos o si hay una cola larga hacia procedimientos más extensos.


In [ ]:
ggplot(datos_limpios, aes(x = duracion_minutos)) +
  geom_histogram(aes(y = after_stat(density)), bins = 30, fill = "steelblue", alpha = 0.7) +
  geom_density(color = "darkblue", linewidth = 1) +
  labs(
    title = "Distribución de duracion_minutos",
    x = "Duración (minutos)", y = "Densidad"
  ) +
  theme_minimal()


La distribución es **asimétrica hacia la derecha**: la mayoría de exámenes dura menos de 30 minutos (laboratorios y radiografías), pero existe una cola de procedimientos más largos (resonancias, tomografías). Esto implica que la relación con el costo podría no ser uniforme en todos los tipos de examen.


### 3.4.1 — Distribución de costo_examen

El costo del examen es la variable que el futuro modelo intentará predecir. Su distribución nos indica si los precios son homogéneos o si hay un grupo de exámenes de alto costo.


In [ ]:
ggplot(datos_limpios, aes(x = costo_examen)) +
  geom_histogram(bins = 30, fill = "darkorange", alpha = 0.7) +
  scale_x_continuous(labels = scales::comma) +
  labs(
    title = "Distribución de costo_examen",
    x = "Costo (COP)", y = "Frecuencia"
  ) +
  theme_minimal()


El costo también presenta **asimetría positiva**: la mayoría de exámenes cuesta entre 80.000 y 200.000 COP, con un grupo minoritario de procedimientos de alto costo. Esta asimetría sugiere que el equipo de modelado podría evaluar una transformación logarítmica del costo en la siguiente entrega.


### 3.4.1 — Distribución de tipo_examen

Analizamos qué tipos de examen concentran la demanda de la clínica, lo que orienta la planificación de capacidad operativa.


In [ ]:
datos_limpios %>%
  count(tipo_examen) %>%
  ggplot(aes(x = reorder(tipo_examen, n), y = n)) +
  geom_col(fill = "seagreen") +
  coord_flip() +
  labs(title = "Frecuencia por tipo de examen", x = "Tipo", y = "Cantidad") +
  theme_minimal()


Los **laboratorios** y **ecografías** concentran la mayor parte de la demanda, seguidos por radiografías. Los procedimientos más costosos y largos (resonancias, tomografías) representan una fracción menor del volumen total.


### 3.4.1 — Distribución de canal_remision

El canal por el que llega el paciente influye en la operación y en los acuerdos de facturación de la clínica.


In [ ]:
datos_limpios %>%
  count(canal_remision) %>%
  ggplot(aes(x = reorder(canal_remision, n), y = n)) +
  geom_col(fill = "purple4") +
  coord_flip() +
  labs(title = "Frecuencia por canal de remisión", x = "Canal", y = "Cantidad") +
  theme_minimal()


El canal **EPS** es el principal origen de pacientes, seguido por **particular** y **remitido**. La medicina prepagada representa una porción menor pero relevante del volumen.


### 3.4.2 — Relación duración vs costo (anticipo de regresión)

Este gráfico de dispersión es el análisis central: muestra si a mayor duración del examen corresponde un mayor costo, que es la hipótesis del futuro modelo de regresión lineal.


In [ ]:
cor_dur_costo <- cor(datos_limpios$duracion_minutos, datos_limpios$costo_examen)
cat("Correlación duración–costo:", round(cor_dur_costo, 3), "\n")

ggplot(datos_limpios, aes(x = duracion_minutos, y = costo_examen)) +
  geom_point(alpha = 0.5, color = "steelblue") +
  geom_smooth(method = "lm", se = TRUE, color = "red") +
  scale_y_continuous(labels = scales::comma) +
  labs(
    title = "Duración vs Costo del examen",
    subtitle = paste0("Correlación = ", round(cor_dur_costo, 3)),
    x = "Duración (minutos)", y = "Costo (COP)"
  ) +
  theme_minimal()


Se observa una **relación positiva** entre duración y costo: los exámenes más largos tienden a ser más costosos. Sin embargo, hay **dispersión considerable** alrededor de la línea de tendencia, lo que indica que la duración sola no explica todo el costo (el tipo de examen y el canal también influyen).


### 3.4.2 — Satisfacción vs días de espera

Exploramos si los pacientes que esperan más días para su cita tienden a calificar peor el servicio, un indicador clave de calidad percibida.


In [ ]:
ggplot(datos_limpios, aes(x = dias_espera_cita, y = calificacion_satisfaccion)) +
  geom_point(alpha = 0.5, color = "coral") +
  geom_smooth(method = "lm", se = TRUE, color = "darkred") +
  labs(
    title = "Satisfacción vs Días de espera",
    x = "Días de espera", y = "Calificación (1-5)"
  ) +
  theme_minimal()


La tendencia sugiere que **mayor tiempo de espera se asocia con menor satisfacción**. Aunque la relación es moderada, reducir los días de espera podría mejorar la experiencia del paciente sin necesidad de modificar precios.


### 3.4.2 — Costo vs tipo de examen

Comparamos el costo medio y la variabilidad entre tipos de examen para entender qué procedimientos generan mayor facturación.


In [ ]:
ggplot(datos_limpios, aes(x = tipo_examen, y = costo_examen)) +
  geom_boxplot(fill = "lightblue") +
  scale_y_continuous(labels = scales::comma) +
  labs(title = "Costo por tipo de examen", x = "Tipo", y = "Costo (COP)") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))


Las **resonancias** y **tomografías** tienen costos medianos y máximos superiores a laboratorios y radiografías. Esto confirma que el tipo de examen es un factor determinante del costo, más allá de la duración.


### 3.4.3 — Matriz de correlación

Calculamos la correlación entre todas las variables numéricas para identificar relaciones lineales y validar la fuerza de la asociación duración–costo.


In [ ]:
vars_corr <- datos_limpios %>%
  select(
    edad, dias_espera_cita, duracion_minutos,
    costo_examen, calificacion_satisfaccion, reingreso_30dias
  )

mat_corr <- cor(vars_corr, use = "complete.obs")
print(round(mat_corr, 3))

corrplot(
  mat_corr,
  method = "color",
  type = "upper",
  addCoef.col = "black",
  number.cex = 0.7,
  tl.cex = 0.8,
  title = "Matriz de correlación",
  mar = c(0, 0, 2, 0)
)


La correlación entre **duracion_minutos** y **costo_examen** es positiva y es la más relevante para el futuro modelo. Las demás correlaciones son débiles o moderadas. Esto confirma que duración es un predictor razonable del costo, aunque no explica toda la variabilidad (hay otros factores como tipo de examen y canal).


### Hallazgos para la dirección de VitalPlus

1. **Duración y costo van de la mano, pero no perfectamente.** Los exámenes más largos tienden a ser más costosos, lo que respalda la idea de estimar costos según duración. Sin embargo, hay variabilidad: dos exámenes con similar duración pueden tener costos distintos según el tipo de procedimiento.

2. **El tiempo de espera impacta la satisfacción.** Pacientes que esperan más días para su cita tienden a calificar peor el servicio. Reducir tiempos de espera podría mejorar la experiencia sin necesidad de reducir costos.

3. **La demanda se concentra en exámenes rápidos y canales EPS.** Laboratorios y radiografías dominan el volumen, y el canal EPS es el principal origen de pacientes. La planificación de capacidad debería priorizar estos servicios en las sedes con mayor demanda.


## 3.5 — Cierre

### ¿Los datos están listos para el modelo de regresión duración → costo?

**Sí, con reservas.** El pipeline de limpieza eliminó duplicados, unificó categorías, imputó nulos y winsorizó outliers en duración y costo. El EDA muestra una relación positiva entre ambas variables. Sin embargo, la dispersión alrededor de la tendencia y la asimetría en el costo indican que un modelo lineal simple podría no capturar toda la variabilidad.

### Advertencias al equipo de modelado

1. **Asimetría en costo:** Evaluar transformación logarítmica de `costo_examen` para estabilizar varianza.
2. **No-linealidad:** La relación duración–costo podría variar por tipo de examen; considerar variables categóricas o modelos con interacciones.
3. **Supuestos de regresión:** Verificar linealidad, homocedasticidad y normalidad de residuos antes de confiar en intervalos de predicción.
4. **Winsorización aplicada:** Los valores extremos ya fueron recortados; documentar esto al interpretar predicciones en el rango alto de costos.
